# Ensemble methods. Exercises


In this section we have only two exercise:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method. 

In [4]:
%store -r data_set
%store -r labels
%store -r test_data_set
%store -r test_labels
%store -r unique_labels

## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [14]:
import numpy as np
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

In [15]:
def build_classifiers():    
    classifiers_classes = [LinearRegression, KNeighborsClassifier, SVC, DecisionTreeClassifier, GaussianNB]
    return [cls().fit(data_set, labels) for cls in classifiers_classes]

In [19]:
def build_stacked_classifier(classifiers, meta_classifier):
    output = []
    for classifier in classifiers:
        output.append(classifier.predict(data_set))
    output = np.array(output).reshape((130,3))
    
    # stacked classifier part:
    stacked_classifier = meta_classifier # set here
    stacked_classifier.fit(output.reshape((130,3)), labels.reshape((130,)))
    test_set = []
    for classifier in classifiers:
        test_set.append(classifier.predict(test_data_set))
    test_set = np.array(test_set).reshape((len(test_set[0]),3))
    predicted = stacked_classifier.predict(test_set)
    return predicted

In [30]:
from itertools import combinations
from copy import deepcopy
import pandas as pd

classifiers = build_classifiers()
results = []
num_classifiers = len(classifiers)
index_combinations = combinations(range(num_classifiers), 3)
classifier_combinations = []
for idx_combo in index_combinations:
    classifier_combo = tuple(classifiers[i] for i in idx_combo)
    classifier_combinations.append(classifier_combo)

results = [
    (*combo, deepcopy(meta), accuracy_score(test_labels, build_stacked_classifier(combo, deepcopy(meta))))
    for combo in classifier_combinations
    for meta in classifiers
    if not isinstance(meta, LinearRegression)
]
            
results.sort(key=lambda x:x[-1], reverse=True)
df = pd.DataFrame(
    results,
    columns=[*range(3), 'meta_classifier', 'accuracy']
)
df.head(10)

,0,1,2,meta_classifier,accuracy
0,LinearRegression(),KNeighborsClassifier(),SVC(),GaussianNB(),1.0
1,LinearRegression(),KNeighborsClassifier(),DecisionTreeClassifier(),GaussianNB(),1.0
2,LinearRegression(),KNeighborsClassifier(),GaussianNB(),GaussianNB(),1.0
3,LinearRegression(),SVC(),DecisionTreeClassifier(),GaussianNB(),1.0
4,LinearRegression(),SVC(),GaussianNB(),GaussianNB(),1.0
5,LinearRegression(),DecisionTreeClassifier(),GaussianNB(),GaussianNB(),1.0
6,KNeighborsClassifier(),SVC(),DecisionTreeClassifier(),GaussianNB(),1.0
7,KNeighborsClassifier(),SVC(),GaussianNB(),GaussianNB(),1.0
8,KNeighborsClassifier(),DecisionTreeClassifier(),GaussianNB(),GaussianNB(),1.0
9,SVC(),DecisionTreeClassifier(),GaussianNB(),GaussianNB(),1.0


## Exercise 2: 

Use the boosting method and change the code to fullfilt the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [31]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# prepare data set

def generate_data(sample_number, feature_number, label_number):
    data_set = np.random.random_sample((sample_number, feature_number))
    labels = np.random.choice(label_number, sample_number)
    return data_set, labels

labels = 2
dimension = 2
test_set_size = 1000
train_set_size = 5000
train_set, train_labels = generate_data(train_set_size, dimension, labels)
test_set, test_labels = generate_data(test_set_size, dimension, labels)

# init weights
number_of_iterations = 10
weights = np.ones((test_set_size,)) / test_set_size


def train_model(classifier, weights):
    return classifier.fit(X=test_set, y=test_labels, sample_weight=weights)

def calculate_error(model):
    predicted = model.predict(test_set)
    I=calculate_accuracy_vector(predicted, test_labels)
    Z=np.sum(I)
    return (1+Z)/1.0

Fill the two functions below:

In [32]:
def set_new_weights(model):
    predictions = model.predict(test_set)
    correct_predictions = predictions == test_labels
    predictions_vector = correct_predictions.astype(int)
    return (1 + predictions_vector) / (1 + predictions_vector).sum()

Train the classifier with the code below:

In [33]:
classifier = DecisionTreeClassifier(max_depth=1, random_state=1)
classifier.fit(X=train_set, y=train_labels)
alphas = []
classifiers = []
for iteration in range(number_of_iterations):
    model = train_model(classifier, weights)
    weights = set_new_weights(model)
    classifiers.append(model)

print(weights)


validate_x, validate_label = generate_data(1, dimension, labels)

[0.00066007 0.00066007 0.00066007 0.00132013 0.00132013 0.00132013
 0.00132013 0.00066007 0.00066007 0.00132013 0.00132013 0.00132013
 0.00066007 0.00066007 0.00066007 0.00132013 0.00066007 0.00132013
 0.00066007 0.00132013 0.00132013 0.00066007 0.00132013 0.00066007
 0.00132013 0.00066007 0.00132013 0.00066007 0.00132013 0.00132013
 0.00132013 0.00066007 0.00066007 0.00066007 0.00132013 0.00066007
 0.00066007 0.00132013 0.00066007 0.00132013 0.00132013 0.00132013
 0.00132013 0.00066007 0.00066007 0.00066007 0.00066007 0.00132013
 0.00066007 0.00132013 0.00066007 0.00132013 0.00066007 0.00066007
 0.00066007 0.00066007 0.00132013 0.00066007 0.00132013 0.00066007
 0.00066007 0.00132013 0.00066007 0.00066007 0.00132013 0.00066007
 0.00132013 0.00132013 0.00066007 0.00132013 0.00066007 0.00132013
 0.00132013 0.00132013 0.00066007 0.00132013 0.00132013 0.00132013
 0.00132013 0.00132013 0.00132013 0.00066007 0.00066007 0.00132013
 0.00066007 0.00066007 0.00132013 0.00132013 0.00132013 0.0006

Set the validation data set:

In [38]:
validate_x, validate_label = generate_data(1, dimension, labels)

Fill the prediction code:

In [41]:
def get_prediction(x):
    predictions = np.array([classifier.predict(x) for classifier in classifiers])
    return [
        np.argmax(np.bincount(predictions[:, i]))
        for i in range(len(x))
    ]

Test it:

In [42]:
prediction = get_prediction(validate_x)[0]

print(prediction)

0
